# 01 · Ngưỡng tốc độ "vật lý bất khả thi"

**Câu hỏi:** Tốc độ giữa hai điểm GPS liên tiếp lớn tới mức nào thì chắc chắn là lỗi, không thể là di chuyển thật?

**Quyết định:** `CleaningThresholds.impossible_speed_kmh = 1100`. Pipeline dùng ngưỡng này ở ba chỗ:
- chọn ứng viên khi trùng timestamp (notebook 02);
- loại điểm gai;
- cắt segment tại bước nhảy bất khả thi.

**Dữ liệu:** 69 user có `labels.txt` (nhãn phương tiện do người dùng ghi). Tốc độ chỉ tính giữa hai điểm liên tiếp **trong cùng một file `.plt`**. Nếu nối điểm cuối file A với điểm đầu file B sẽ sinh ra tốc độ giả; ví dụ user 062: hai file cách nhau 466 km trong 24 phút cho ra 1,179 km/h giả.

**Lưu ý:** bản thân dữ liệu có nhãn cũng chứa lỗi GPS, nên không lấy `max` làm căn cứ; phải soi từng điểm nghi ngờ.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK = "01_speed_threshold"
DATA_DIR = Path("../data/raw")
OUT_DIR = Path("outputs") / NOTEBOOK
FIG_DIR, TABLE_DIR, CACHE_DIR, MAP_DIR = (OUT_DIR / d for d in ("figures", "tables", "cache", "maps"))
for d in (FIG_DIR, TABLE_DIR, CACHE_DIR, MAP_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(Path("../src").resolve()))
EARTH_RADIUS_KM = 6371.0088
RAW_COLUMNS = ["lat", "lon", "reserved", "altitude", "date_days", "date_str", "time_str"]


def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized haversine distance (km) between paired coordinate arrays."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def load_user_trajectory(user_id: str) -> pd.DataFrame | None:
    """All .plt files of a user, keeping the source file so speeds stay within one file."""
    files = sorted((DATA_DIR / user_id / "Trajectory").glob("*.plt"))
    if not files:
        return None
    parts = [pd.read_csv(f, skiprows=6, header=None, names=RAW_COLUMNS).assign(source_file=f.name) for f in files]
    traj = pd.concat(parts, ignore_index=True)
    traj["timestamp"] = pd.to_datetime(traj["date_str"] + " " + traj["time_str"],
                                       format="%Y-%m-%d %H:%M:%S", errors="coerce")
    return traj.dropna(subset=["timestamp"]).reset_index(drop=True)


def load_labels(user_id: str) -> pd.DataFrame | None:
    """labels.txt: tab-separated Start Time / End Time / Transportation Mode."""
    path = DATA_DIR / user_id / "labels.txt"
    if not path.exists():
        return None
    df = pd.read_csv(path, sep="\t", skiprows=1, header=None, names=["start", "end", "mode"])
    df["start_dt"] = pd.to_datetime(df["start"], format="%Y/%m/%d %H:%M:%S", errors="coerce")
    df["end_dt"] = pd.to_datetime(df["end"], format="%Y/%m/%d %H:%M:%S", errors="coerce")
    return df.dropna(subset=["start_dt", "end_dt"])[["start_dt", "end_dt", "mode"]].reset_index(drop=True)


def assign_mode(timestamps: pd.Series, labels: pd.DataFrame) -> np.ndarray:
    """Transport mode of each point from the labelled [start, end] intervals ("unknown" otherwise)."""
    modes = np.full(len(timestamps), "unknown", dtype=object)
    ts = timestamps.to_numpy()
    for start, end, mode in labels.itertuples(index=False):
        modes[(ts >= np.datetime64(start)) & (ts <= np.datetime64(end))] = mode
    return modes


def point_speeds(traj: pd.DataFrame) -> pd.DataFrame:
    """Distance / time / speed from the previous point of the SAME file (NaN for a file's first point)."""
    out = traj.sort_values(["source_file", "timestamp"], kind="stable").reset_index(drop=True)
    lat, lon = out["lat"].to_numpy(), out["lon"].to_numpy()
    dist_km = np.full(len(out), np.nan)
    dist_km[1:] = haversine_km(lat[:-1], lon[:-1], lat[1:], lon[1:])
    dt_s = out["timestamp"].diff().dt.total_seconds().to_numpy(copy=True)
    new_file = (out["source_file"] != out["source_file"].shift(1)).to_numpy()
    dist_km[new_file] = np.nan
    dt_s[new_file] = np.nan
    out["distance_km"] = dist_km
    out["delta_time_s"] = dt_s
    out["speed_kmh"] = dist_km / np.where(dt_s <= 0, np.nan, dt_s) * 3600
    return out

## 1. Thu thập tốc độ có nhãn phương tiện
Chỉ giữ những điểm có tốc độ hợp lệ (bỏ điểm đầu mỗi file và các cặp có `dt <= 0`) và nằm trong một khoảng nhãn. Kết quả được cache, vì lần chạy đầu mất vài phút.

In [ ]:
SPEEDS_PATH = CACHE_DIR / "labeled_point_speeds.parquet"
if SPEEDS_PATH.exists():
    speed_df = pd.read_parquet(SPEEDS_PATH)
else:
    users = sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir())
    labeled_users = [u for u in users if (DATA_DIR / u / "labels.txt").exists()]
    rows = []
    for i, user_id in enumerate(labeled_users, start=1):
        print(f"\r[{i}/{len(labeled_users)}] user {user_id}", end="", flush=True)
        traj, labels = load_user_trajectory(user_id), load_labels(user_id)
        if traj is None or labels is None:
            continue
        sp = point_speeds(traj)
        sp["mode"] = assign_mode(sp["timestamp"], labels)
        sp["user_id"] = user_id
        rows.append(sp.loc[sp["speed_kmh"].notna() & (sp["mode"] != "unknown"),
                           ["user_id", "source_file", "timestamp", "mode", "distance_km", "delta_time_s", "speed_kmh"]])
    print()
    speed_df = pd.concat(rows, ignore_index=True)
    speed_df.to_parquet(SPEEDS_PATH)

print(f"Labelled points with a valid speed: {len(speed_df):,} from {speed_df['user_id'].nunique()} users")
speed_df["mode"].value_counts()

## 2. Phân phối tốc độ theo phương tiện
Dùng percentile thay vì `max`, vì `max` bị chính các lỗi GPS chi phối. Bảng cuối dùng tốc độ trung bình của từng phương tiện trong User Guide GeoLife 1.3 để kiểm tra độ hợp lý.

In [ ]:
def summarize(s: pd.Series) -> pd.Series:
    q = s.quantile([0.5, 0.9, 0.95, 0.99, 0.999, 0.9999])
    return pd.Series({"count": len(s), "mean": s.mean(), "p50": q[0.5], "p90": q[0.9], "p95": q[0.95],
                      "p99": q[0.99], "p99_9": q[0.999], "p99_99": q[0.9999], "max": s.max()})

speed_stats = (speed_df.groupby("mode")["speed_kmh"].apply(summarize).unstack()
               .sort_values("p99_9", ascending=False))
speed_stats.to_csv(TABLE_DIR / "speed_stats_by_mode.csv")

# User Guide 1.3: total distance / total duration over the whole dataset (sanity check only)
user_guide_avg_kmh = pd.Series({"walk": 10123 / 5460, "bike": 6495 / 2410, "bus": 20281 / 1507,
                                "car": 32866 / 2384, "train": 36253 / 745, "airplane": 24789 / 40},
                               name="user_guide_avg_kmh")
speed_stats.join(user_guide_avg_kmh).round(1)

In [ ]:
modes = speed_stats.index.tolist()
cap = speed_df["speed_kmh"].quantile(0.999)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
axes[0].boxplot([speed_df.loc[speed_df["mode"] == m, "speed_kmh"].clip(upper=cap) for m in modes],
                tick_labels=modes, showfliers=False)
axes[0].set_title("Speed distribution by transport mode (clipped at global P99.9)")
axes[0].set_ylabel("Speed (km/h)")
axes[0].tick_params(axis="x", rotation=45)
axes[1].bar(modes, speed_stats["p99_9"], color="#3b6ea5")
axes[1].set_title("P99.9 speed by transport mode")
axes[1].set_ylabel("Speed (km/h)")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig01_speed_distribution_by_mode.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Soi bằng chứng: các điểm tàu và máy bay trên 950 km/h

Mục đích là tách **điểm nhanh thật** (máy bay đang bay ổn định) khỏi **điểm lỗi** (GPS nhảy điểm). Không lấy percentile gộp tàu và máy bay, vì dữ liệu tàu chứa lỗi (không đoàn tàu nào chạy được 4,678 km/h).

In [ ]:
fast = (speed_df[speed_df["mode"].isin(["train", "airplane"]) & (speed_df["speed_kmh"] > 950)]
        .sort_values("speed_kmh", ascending=False))
fast.to_csv(TABLE_DIR / "fast_mode_points_over_950kmh.csv", index=False)
print(f"Train/airplane points above 950 km/h: {len(fast)}")
print(fast.groupby(["user_id", "mode"]).size().rename("n_points").to_string())
fast.head(25)[["user_id", "mode", "timestamp", "source_file", "delta_time_s", "distance_km", "speed_kmh"]].round(
    {"distance_km": 2, "speed_kmh": 2})

**Đọc kết quả:**

- **"train" trên 950 km/h:** `dt` chỉ 2–10 giây, nhảy 1–6 km, trong cùng một file. Đây là GPS nhảy điểm: không đoàn tàu nào chạy nhanh như vậy. Các điểm này tập trung ở một khu vực phía bắc Bắc Kinh (lat 40.06–40.09, lon 116.26–116.33), lặp lại nhiều ngày ở hai user khác nhau.
- **"airplane" của user 128, ngày 2009-02-22:** chuỗi điểm cách đều 10 giây, tốc độ ổn định 975–1,005 km/h, bay về hướng đông. Đây là **dữ liệu bay thật** (tốc độ so với mặt đất có cộng gió xuôi). Cặp điểm 1,048 km/h (khoảng ngắt 22 phút, 390 km, cùng chuyến bay) cũng hợp lý.
- **"airplane" 1,879 km/h** (khoảng ngắt 38 phút, 1,184 km): tốc độ trung bình trong khoảng ngắt vượt quá tốc độ tối đa của máy bay chở khách, nên đây là **lỗi**.

## 4. Chọn ngưỡng
Ngưỡng được neo theo hai mốc đã kiểm tra trực tiếp: tốc độ **thật** cao nhất và tốc độ **lỗi** thấp nhất.

In [ ]:
REAL_MAX_KMH = 1048.1    # airplane, user 128: 2009-02-22 04:22:59 -> 04:45:17 (1,338 s, 389.55 km)
ERROR_MIN_KMH = 1122.7   # "train", user 128: 2007-04-18 14:42:26 -> 14:42:36 (10 s, 3.12 km)
IMPOSSIBLE_KMH = 1100

airplane = speed_df.loc[speed_df["mode"] == "airplane", "speed_kmh"]
train = speed_df.loc[speed_df["mode"] == "train", "speed_kmh"]
print(f"Airplane: n={len(airplane):,}  P99={airplane.quantile(0.99):.1f}  "
      f"P99.9={airplane.quantile(0.999):.1f}  max={airplane.max():.1f} km/h")
print(f"Highest verified REAL speed : {REAL_MAX_KMH} km/h")
print(f"Lowest verified ERROR speed : {ERROR_MIN_KMH} km/h")
print(f"Chosen threshold            : {IMPOSSIBLE_KMH} km/h")
print(f"Labelled points above it    : airplane {(airplane > IMPOSSIBLE_KMH).sum()}, "
      f"train {(train > IMPOSSIBLE_KMH).sum()} (all verified errors, see section 3)")

## 5. Vì sao bỏ bộ lọc 180 km/h

Pipeline cũ có thêm bước xoá mọi điểm trên 180 km/h, để lọc nhiễu multipath. Ngưỡng này không có căn cứ; bảng dưới cho thấy nó xoá bao nhiêu điểm **có nhãn thật** ở từng phương tiện. Bước này đã bị bỏ khi dự án gộp về **một pipeline duy nhất**: không bài toán nào trong phạm vi cần tốc độ từng điểm, và stay-point vốn không chứa điểm di chuyển nhanh.

In [ ]:
over_180 = (speed_df.assign(over_180=speed_df["speed_kmh"] > 180).groupby("mode")["over_180"]
            .agg(n_points="size", n_over_180="sum"))
over_180["share_over_180_pct"] = 100 * over_180["n_over_180"] / over_180["n_points"]
over_180 = over_180.sort_values("share_over_180_pct", ascending=False)
over_180.to_csv(TABLE_DIR / "share_over_180kmh_by_mode.csv")

GROUND_MODES = ["walk", "run", "bike", "bus", "car", "taxi", "motorcycle"]
ground_p999 = speed_df.loc[speed_df["mode"].isin(GROUND_MODES), "speed_kmh"].quantile(0.999)
print(f"Ground-mode P99.9 speed (typical GPS drift level): {ground_p999:.1f} km/h")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(over_180.index, over_180["share_over_180_pct"], color="#c0504d")
ax.set_title("Share of labelled points a 180 km/h filter would delete")
ax.set_ylabel("Points above 180 km/h (%)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig02_share_over_180kmh_by_mode.png", dpi=150, bbox_inches="tight")
plt.show()
over_180.round(2)

## 6. Nhiễu GPS và tốc độ thực tế của phương tiện

**Vì sao:** mentor nhận xét 20% dữ liệu có nhãn (4.86 triệu điểm) là đủ để lấy tốc độ theo phương tiện, và quy tắc xử lý trùng timestamp còn tính ngẫu nhiên. Mục này trả lời ba câu:
1. Nhiễu vị trí GPS lớn cỡ nào?
2. Trừ nhiễu đi, tốc độ trong dữ liệu có khớp với tốc độ thực tế của từng phương tiện không?
3. Có nên thay ngưỡng chung 1100 km/h bằng ngưỡng riêng cho từng phương tiện không?

**Nguyên tắc:** chỉ dùng **điểm GPS thật**, không nội suy. Tốc độ thực tế lấy từ **nguồn bên ngoài**, không lấy từ dữ liệu:

| Phương tiện | Thường gặp (km/h) | Tối đa thực tế (km/h) | Nguồn |
|---|---|---|---|
| walk | 4.8–6.4 | 8 (đi nhanh) | Bohannon & Andrews 2011; Wikipedia *Walking* |
| run | 8–10 | 20 | Strava / Marathon Handbook |
| bike | 13.5–22 (đô thị) | 40 | nghiên cứu Lyon (arXiv 1011.6266), Copenhagen |
| bus | 15–20 (Bắc Kinh) | 100 (luật: xe không phải xe con trên cao tốc) | ESMAP Beijing Transport; luật GTĐB Trung Quốc |
| car, taxi | 25–28 (giờ cao điểm Bắc Kinh) | 120 (luật: xe con trên cao tốc) | Lincoln Institute; Wikipedia *Speed limits in China* |
| motorcycle | – | 80 (luật trên cao tốc) | luật GTĐB Trung Quốc |
| subway | – | 80–110 (Airport Express 110) | Wikipedia *Beijing Subway rolling stock* |
| train | 120–200 | 350 (Bắc Kinh–Thiên Tân, 8/2008) | Wikipedia *Beijing–Tianjin intercity railway* |
| airplane | 850–950 | 1,100–1,327 (có gió xuôi) | ScienceAlert; LiveScience |

**Dữ liệu:** output đã làm sạch (`data/processed/users`, đã bỏ trùng và điểm gai), các **bộ ba điểm thật** `(i−k, i, i+k)` cùng file và cùng khoảng nhãn, với `k` = 1, 5, 10. Với `k` = 5 và 10, hai điểm ngoài cách nhau giống **điểm neo** thật trong các nhóm trùng (trung vị 15 s mỗi phía).

In [ ]:
CLEAN_DIR = Path("../data/processed/users")
REGIME = {"walk": "slow", "run": "slow", "bike": "slow", "bus": "road", "car": "road", "taxi": "road",
          "motorcycle": "road", "subway": "road", "boat": "road", "train": "train", "airplane": "airplane"}
REGIMES = ["slow", "road", "train", "airplane"]
TRIPLES_PATH = CACHE_DIR / "labeled_triples_clean.parquet"

if TRIPLES_PATH.exists():
    triples = pd.read_parquet(TRIPLES_PATH)
else:
    parts = []
    for path in sorted(CLEAN_DIR.glob("user_*.parquet")):
        uid = path.stem.split("_")[1]
        labels = load_labels(uid)
        if labels is None:
            continue
        labels = labels.sort_values("start_dt").reset_index(drop=True)
        df = pd.read_parquet(path, columns=["source_file", "datetime", "lat", "lon"])
        df = df.sort_values(["source_file", "datetime"], kind="stable").reset_index(drop=True)
        ts = df["datetime"].to_numpy()
        k_lab = np.searchsorted(labels["start_dt"].to_numpy(), ts, side="right") - 1
        inside = k_lab >= 0
        inside[inside] = ts[inside] <= labels["end_dt"].to_numpy()[k_lab[inside]]
        lab_id = np.where(inside, k_lab, -1)
        f, lat, lon = df["source_file"].to_numpy(), df["lat"].to_numpy(), df["lon"].to_numpy()
        t = ts.astype("datetime64[ns]").astype(np.int64) / 1e9
        for k in (1, 5, 10):
            i = np.arange(k, len(df) - k)
            a, b = i - k, i + k
            same = (f[a] == f[i]) & (f[i] == f[b]) & (lab_id[i] >= 0) & (lab_id[a] == lab_id[i]) & (lab_id[i] == lab_id[b])
            i, a, b = i[same], a[same], b[same]
            dt_in, dt_out = t[i] - t[a], t[b] - t[i]
            parts.append(pd.DataFrame({
                "user_id": uid, "k": k, "mode": labels["mode"].to_numpy()[lab_id[i]],
                "dt_in": dt_in, "dt_out": dt_out,
                "v_in": haversine_km(lat[a], lon[a], lat[i], lon[i]) / dt_in * 3600,
                "v_out": haversine_km(lat[i], lon[i], lat[b], lon[b]) / dt_out * 3600,
                "v_anchor": haversine_km(lat[a], lon[a], lat[b], lon[b]) / (dt_in + dt_out) * 3600,
            }))
    triples = pd.concat(parts, ignore_index=True)
    triples.to_parquet(TRIPLES_PATH)

triples["regime"] = triples["mode"].map(REGIME)
print(f"Real labelled triples: {len(triples):,} from {triples['user_id'].nunique()} users")
print("Median time between the two anchors (s) by k:", triples.groupby("k")["dt_in"].median().mul(2).to_dict())

### 6.1 Nhiễu GPS
Lấy các điểm thật cách nhau đúng **1 giây**. Trong 1 giây, người đi bộ chỉ đi được khoảng 1.4 m (5 km/h), nên phần lệch lớn hơn là **sai số định vị**. Đo cho cả các phương tiện khác để xem nhiễu có phụ thuộc phương tiện không.

In [ ]:
step1 = triples[(triples["k"] == 1) & (triples["dt_in"] == 1)].copy()
step1["d_m"] = step1["v_in"] / 3.6
noise = step1.groupby("mode")["d_m"].quantile([0.5, 0.99, 0.999]).unstack()
noise.columns = ["p50_m", "p99_m", "p99_9_m"]
noise["n"] = step1.groupby("mode").size()
noise = noise[noise["n"] >= 1000].sort_values("p99_9_m")
noise.to_csv(TABLE_DIR / "gps_noise_one_second_steps.csv")
GPS_NOISE_M = float(noise.loc["walk", "p99_9_m"])
print(f"GPS noise margin E = P99.9 of 1-s walking steps = {GPS_NOISE_M:.1f} m")
noise.round(1)

**Đọc kết quả:** với mọi phương tiện, P99.9 khoảng cách trong 1 giây đều khoảng **29–34 m**, dù đi bộ chỉ đi được khoảng 1.4 m. Nhiễu GPS là cỡ **vài chục mét**, gần như **không phụ thuộc phương tiện**. Ở `dt` = 1 s, nhiễu đó đã thành "tốc độ" 100–120 km/h; vì vậy tốc độ một bước không cho biết người dùng đi nhanh thế nào.

### 6.2 Trừ nhiễu rồi so với tốc độ thực tế
Tốc độ đã trừ nhiễu = `max(0, khoảng cách − E) / dt`, đo trên khoảng của điểm neo (`k` = 5, 10) để nhiễu ít ảnh hưởng. `share_over_web_pct` là tỉ lệ vượt mức tối đa thực tế ở bảng trên.

In [ ]:
WEB_UPPER_KMH = {"walk": 8, "run": 20, "bike": 40, "bus": 100, "car": 120, "taxi": 120, "motorcycle": 80,
                 "subway": 110, "train": 350, "airplane": 1100}
wide = triples[triples["k"].isin([5, 10])]
corrected = np.maximum(0, wide["v_in"] / 3.6 * wide["dt_in"] - GPS_NOISE_M) / wide["dt_in"] * 3.6
vs_web = corrected.groupby(wide["mode"]).quantile([0.5, 0.95, 0.99, 0.999]).unstack()
vs_web.columns = ["p50", "p95", "p99", "p99_9"]
vs_web["web_upper_kmh"] = pd.Series(WEB_UPPER_KMH)
vs_web["share_over_web_pct"] = [100 * (corrected[wide["mode"] == m] > WEB_UPPER_KMH.get(m, np.inf)).mean()
                                for m in vs_web.index]
vs_web.to_csv(TABLE_DIR / "corrected_speed_vs_real_world.csv")
vs_web.sort_values("p50").round(2)

**Đọc kết quả:**
- Trừ nhiễu xong, tốc độ trong dữ liệu **khớp với tốc độ thực tế**: mọi phương tiện chỉ có 0–0.5% số điểm vượt mức tối đa.
- Ngoại lệ là **walk**: khoảng 2.4% vượt 8 km/h, và P99.9 còn khoảng 60 km/h. Nhiều khả năng đó là **nhãn sai** (người dùng lên xe mà không đổi nhãn), không phải người đi bộ.
- **Máy bay:** nguồn bên ngoài cho thấy gió xuôi có thể đẩy tốc độ so với mặt đất lên 1,100–1,327 km/h, tức **trên** ngưỡng 1100. Trong GeoLife, tốc độ thật cao nhất đã kiểm chứng là 1,048 km/h (mục 3), nên ngưỡng chưa cắt nhầm điểm nào. Nhưng nếu dữ liệu có chuyến bay đường dài với gió xuôi mạnh, ngưỡng này có thể phải nâng lên.

### 6.3 Có nên dùng ngưỡng riêng cho từng phương tiện?
Thử quy tắc vật lý: **khoảng cách tối đa = tốc độ tối đa thực tế × `dt` + E**. Phương tiện được đoán từ tốc độ đã trừ nhiễu giữa hai điểm neo: chọn nhóm chậm nhất có tốc độ **thường gặp** ≥ tốc độ đó. Những phương tiện có tốc độ thường gặp trùng nhau thì gộp lại, vì tốc độ không phân biệt được chúng:
- `bike_bus`: 13.5–22 so với 15–20 km/h;
- `car_taxi_subway_moto`: cùng mức đường phố.

Điểm i là điểm **thật**, nên mọi lần bị loại đều là loại nhầm. Mục tiêu là ≤ 0.1%.

In [ ]:
GROUPS = [("walk", 6.4, 8, ["walk"]), ("run", 10, 20, ["run"]), ("bike_bus", 22, 100, ["bike", "bus"]),
          ("car_taxi_subway_moto", 60, 120, ["car", "taxi", "subway", "motorcycle"]),
          ("train", 160, 350, ["train"]), ("airplane", np.inf, 1100, ["airplane"])]
typical = np.array([g[1] for g in GROUPS], float)
upper = np.array([g[2] for g in GROUPS], float)
group_of = {m: i for i, g in enumerate(GROUPS) for m in g[3]}

rows = []
for k in (1, 5, 10):
    d = triples[(triples["k"] == k) & triples["mode"].isin(group_of)]
    dt_anchor = d["dt_in"] + d["dt_out"]
    v_anchor = np.maximum(0, d["v_anchor"] / 3.6 * dt_anchor - GPS_NOISE_M) / dt_anchor * 3.6
    guess = np.minimum(np.searchsorted(typical, v_anchor.to_numpy(), side="left"), len(GROUPS) - 1)
    limit_m_in = upper[guess] / 3.6 * d["dt_in"].to_numpy() + GPS_NOISE_M
    limit_m_out = upper[guess] / 3.6 * d["dt_out"].to_numpy() + GPS_NOISE_M
    rejected = (d["v_in"].to_numpy() / 3.6 * d["dt_in"].to_numpy() > limit_m_in) | \
               (d["v_out"].to_numpy() / 3.6 * d["dt_out"].to_numpy() > limit_m_out)
    truth = d["mode"].map(group_of).to_numpy()
    rows.append({"k": k, "false_reject_pct": 100 * rejected.mean(), "group_guess_accuracy_pct": 100 * (guess == truth).mean(),
                 **{g[0]: 100 * rejected[truth == i].mean() for i, g in enumerate(GROUPS)}})
physical_rule = pd.DataFrame(rows)
physical_rule.to_csv(TABLE_DIR / "physical_limit_false_rejection.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
for g in [g[0] for g in GROUPS]:
    ax.plot(physical_rule["k"], physical_rule[g], marker="o", label=g)
ax.axhline(0.1, color="red", ls="--", label="target 0.1%")
ax.set_xticks([1, 5, 10])
ax.set_xlabel("k (anchors are the k-th real point before and after)")
ax.set_ylabel("Real points rejected (%)")
ax.set_title("Physical per-mode limit (real-world upper speed x dt + GPS noise): false rejection")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig03_physical_limit_false_rejection.png", dpi=150)
plt.show()
physical_rule.round(3)

**Đọc kết quả:** với hai điểm liền kề (`k` = 1) thì đạt mục tiêu, nhưng khi điểm neo cách xa như thực tế (`k` = 5, 10), tỉ lệ loại nhầm lên **khoảng 1–2%**, chủ yếu ở bus và car. Nguyên nhân là **đoán phương tiện**, không phải ngưỡng: xe dừng đèn đỏ rồi tăng tốc trong khoảng giữa hai điểm neo, nên tốc độ trung bình thấp, bị đoán là đi bộ, và đoạn xe chạy bị loại.

Một phương án khác, lấy ngưỡng theo P99.9 của dữ liệu, đã được thử: loại nhầm ít, nhưng ngưỡng của đi bộ thành 108.8 km/h. Con số đó phản ánh nhiễu GPS và nhãn sai, không có ý nghĩa vật lý, nên không dùng.

**Kết luận mục 6:**
- **Không** dùng ngưỡng riêng theo phương tiện. Ngưỡng chung 1100 km/h giữ nguyên cho kiểm tra hợp lệ.
- Tính ngẫu nhiên mà mentor nhắc tới được giải quyết ở notebook 02, phần 3, bằng cách **chọn ứng viên theo tốc độ**. Cách đó không cần ngưỡng mới.
- Nhiễu GPS **E ≈ 34 m** được ghi lại để dùng khi cần biên sai số vị trí.

## Kết luận

- **`impossible_speed_kmh = 1100`**: nằm giữa tốc độ thật cao nhất đã kiểm tra (1,048.1 km/h, máy bay) và tốc độ lỗi thấp nhất đã kiểm tra (1,122.7 km/h, tàu nhảy điểm 3 km trong 10 giây). Ngưỡng tách đúng cả hai nhóm đã soi.
- **Giới hạn:** biên an toàn mỏng, và chỉ dựa trên vài user. Lỗi GPS dưới 1,100 km/h (ví dụ tàu nhảy với tốc độ 500 km/h) sẽ lọt qua. Điều này chấp nhận được, vì ngưỡng chỉ nhằm bắt điểm **bất khả thi**, không nhằm bắt mọi lỗi.
- **Bộ lọc 180 km/h đã bị bỏ:** nó xoá 56.7% điểm máy bay và 3.6% điểm tàu có nhãn thật (mục 5).
- **Nhiễu GPS và ngưỡng theo phương tiện** (mục 6): nhiễu vị trí khoảng 34 m (P99.9). Trừ nhiễu xong, tốc độ khớp với tốc độ thực tế của từng phương tiện. Ngưỡng riêng theo phương tiện **không** được dùng, vì đoán phương tiện từ hai điểm neo làm loại nhầm 1–2% điểm thật. Máy bay có gió xuôi có thể vượt 1100 km/h (tới 1,327 km/h); GeoLife chưa có trường hợp nào như vậy.

In [ ]:
decisions = pd.DataFrame([
    {"threshold": "impossible_speed_kmh", "value": IMPOSSIBLE_KMH,
     "basis": f"between highest verified real speed ({REAL_MAX_KMH}) and lowest verified GPS-jump error ({ERROR_MIN_KMH})"},
    {"threshold": "ground_mode_p99_9_kmh", "value": round(float(ground_p999), 1),
     "basis": "typical GPS drift level of ground modes (reference only; drift filter removed)"},
    {"threshold": "gps_noise_margin_m", "value": round(GPS_NOISE_M, 1),
     "basis": "P99.9 displacement of real walking points 1 s apart (section 6.1); reference only"},
])
decisions.to_csv(TABLE_DIR / "threshold_decisions.csv", index=False)
decisions